# Draft-05: Final XGBoost Blend Pipeline (Kaggle)

Score-first update from draft-04: leakage-safe 5-fold CV, two complementary XGBoost models per fold, OOF blend search, and OOF threshold tuning for Macro F1.


In [ ]:
import importlib.util
import os
import random
import subprocess
import sys
import time
from datetime import datetime, UTC
from pathlib import Path
from zipfile import ZipFile

import numpy as np
import pandas as pd
from dotenv import load_dotenv
from IPython.display import display
from kaggle.api.kaggle_api_extended import KaggleApi
from sklearn.compose import ColumnTransformer
from sklearn.metrics import confusion_matrix, f1_score, precision_score, recall_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import OneHotEncoder
from xgboost import XGBClassifier

def run_cmd(cmd):
    print('Running:', ' '.join(cmd))
    subprocess.check_call(cmd)

def ensure_package(module_name, package_name=None):
    pkg = package_name or module_name
    if importlib.util.find_spec(module_name) is not None:
        print(f'{pkg} is already installed')
        return
    run_cmd([sys.executable, '-m', 'pip', 'install', pkg])

ensure_package('xgboost')
ensure_package('dotenv', 'python-dotenv')
ensure_package('kaggle')

pd.set_option('display.max_columns', 200)
pd.set_option('display.float_format', lambda x: f'{x:,.6f}')


In [ ]:
COMPETITION = 'fraud-detection-cpe-232-data-models'
RUN_DOWNLOAD = True
RUN_SUBMISSION = True
USE_GPU_IF_AVAILABLE = True
FULL_REFIT_FOR_SUBMISSION = True

ID_COL = 'id'
LABEL_COL = 'is_fraud'
TIME_COL = 'time_ind'
N_SPLITS = 5
RANDOM_STATE = 42
EPS = 1.0
THRESHOLD_GRID = np.linspace(0.001, 0.999, 999)
BLEND_WEIGHT_GRID = np.linspace(0.0, 1.0, 51)

BASE_MODEL_A = {
    'objective': 'binary:logistic',
    'eval_metric': 'logloss',
    'n_estimators': 2200,
    'learning_rate': 0.04,
    'max_depth': 8,
    'min_child_weight': 5,
    'subsample': 0.9,
    'colsample_bytree': 0.8,
    'reg_lambda': 1.0,
    'gamma': 0.0,
    'random_state': RANDOM_STATE,
    'n_jobs': -1,
    'early_stopping_rounds': 150,
}

BASE_MODEL_B = {
    'objective': 'binary:logistic',
    'eval_metric': 'logloss',
    'n_estimators': 2600,
    'learning_rate': 0.03,
    'max_depth': 6,
    'min_child_weight': 8,
    'subsample': 0.85,
    'colsample_bytree': 0.7,
    'reg_lambda': 2.0,
    'gamma': 1.0,
    'random_state': RANDOM_STATE + 7,
    'n_jobs': -1,
    'early_stopping_rounds': 180,
}

random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

def detect_xgb_device(use_gpu_if_available=True):
    if not use_gpu_if_available:
        return {'tree_method': 'hist'}, 'cpu'
    X_probe = np.array([[0.0], [1.0], [2.0], [3.0]], dtype=np.float32)
    y_probe = np.array([0, 1, 0, 1], dtype=np.int32)
    try:
        probe = XGBClassifier(
            objective='binary:logistic',
            n_estimators=2,
            tree_method='hist',
            device='cuda',
            eval_metric='logloss',
            n_jobs=1,
        )
        probe.fit(X_probe, y_probe, verbose=False)
        return {'tree_method': 'hist', 'device': 'cuda'}, 'gpu'
    except Exception:
        try:
            probe = XGBClassifier(
                objective='binary:logistic',
                n_estimators=2,
                tree_method='gpu_hist',
                predictor='gpu_predictor',
                eval_metric='logloss',
                n_jobs=1,
            )
            probe.fit(X_probe, y_probe, verbose=False)
            return {'tree_method': 'gpu_hist', 'predictor': 'gpu_predictor'}, 'gpu'
        except Exception:
            return {'tree_method': 'hist'}, 'cpu'

DEVICE_PARAMS, DEVICE_MODE = detect_xgb_device(USE_GPU_IF_AVAILABLE)

def resolve_data_paths_fallback():
    kaggle_candidates = [
        Path('/kaggle/input/fraud-detection-cpe-232-data-models'),
        Path('/kaggle/input/fraudulent-transaction-detect'),
        Path('/kaggle/input'),
    ]
    local_candidates = [
        Path.cwd(),
        Path.cwd() / 'kaggle/01-fraudulent-transaction-detect',
        Path('/Users/beam/Workspace/Course/my-cpe-lab/Y2/CPE232/kaggle/01-fraudulent-transaction-detect'),
    ]

    candidate_pairs = []
    for base in kaggle_candidates + local_candidates:
        candidate_pairs.extend([
            (base / 'train.csv', base / 'test.csv', base / 'sample_submission.csv'),
            (base / 'data' / 'train.csv', base / 'data' / 'test.csv', base / 'data' / 'sample_submission.csv'),
            (
                base / 'fraud-detection-cpe-232-data-models' / 'train.csv',
                base / 'fraud-detection-cpe-232-data-models' / 'test.csv',
                base / 'fraud-detection-cpe-232-data-models' / 'sample_submission.csv',
            ),
        ])
    for train_path, test_path, sample_path in candidate_pairs:
        if train_path.exists() and test_path.exists() and sample_path.exists():
            return train_path, test_path, sample_path
    raise FileNotFoundError('Could not find train/test/sample CSV files in Kaggle input or local folders')

def prepare_competition_data(api_client, competition, data_dir, force_download=False):
    data_dir = Path(data_dir)
    data_dir.mkdir(parents=True, exist_ok=True)
    zip_path = data_dir / f'{competition}.zip'
    extract_dir = data_dir / competition

    if force_download or not zip_path.exists():
        files_resp = api_client.competition_list_files(competition)
        try:
            api_client.competition_download_files(
                competition,
                path=str(data_dir),
                force=force_download,
                quiet=False,
            )
            print('Competition zip download complete')
        except Exception as download_error:
            print('Bulk download failed, fallback to per-file download:', download_error)
            for f in files_resp.files:
                file_path = data_dir / f.name
                if file_path.exists() and not force_download:
                    continue
                api_client.competition_download_file(
                    competition=competition,
                    file_name=f.name,
                    path=str(data_dir),
                    force=force_download,
                    quiet=False,
                )

    if zip_path.exists():
        if not extract_dir.exists() or not any(extract_dir.iterdir()):
            extract_dir.mkdir(parents=True, exist_ok=True)
            with ZipFile(zip_path, 'r') as zf:
                zf.extractall(extract_dir)

    train_path = extract_dir / 'train.csv'
    test_path = extract_dir / 'test.csv'
    sample_path = extract_dir / 'sample_submission.csv'

    if not (train_path.exists() and test_path.exists() and sample_path.exists()):
        fallback_train = data_dir / 'train.csv'
        fallback_test = data_dir / 'test.csv'
        fallback_sample = data_dir / 'sample_submission.csv'
        if fallback_train.exists() and fallback_test.exists() and fallback_sample.exists():
            train_path, test_path, sample_path = fallback_train, fallback_test, fallback_sample
        else:
            raise FileNotFoundError('Could not resolve competition train/test/sample files')

    return train_path, test_path, sample_path, zip_path, extract_dir


In [ ]:
load_dotenv('.env', override=True)
os.environ.pop('KAGGLE_API_TOKEN', None)

api = None
try:
    api = KaggleApi()
    api.authenticate()
    print('Authenticated user:', api.get_config_value('username'))
except Exception as auth_error:
    print('Kaggle API auth not ready:', auth_error)
    print('Falling back to existing /kaggle/input or local data if available')

if RUN_DOWNLOAD:
    if api is None:
        raise RuntimeError('RUN_DOWNLOAD=True but Kaggle API auth is not available')
    BASE_DIR = Path.cwd()
    DATA_DIR = BASE_DIR / 'data'
    TRAIN_PATH, TEST_PATH, SAMPLE_PATH, ZIP_PATH, EXTRACT_DIR = prepare_competition_data(
        api_client=api,
        competition=COMPETITION,
        data_dir=DATA_DIR,
        force_download=False,
    )
else:
    TRAIN_PATH, TEST_PATH, SAMPLE_PATH = resolve_data_paths_fallback()
    BASE_DIR = TRAIN_PATH.parent

OUTPUT_DIR = Path('/kaggle/working') if Path('/kaggle/working').exists() else BASE_DIR
OUTPUT_PATH = OUTPUT_DIR / 'submission_draft05_xgboost_kaggle.csv'
DIAGNOSTICS_PATH = OUTPUT_DIR / 'diagnostics_draft05_xgboost_kaggle.csv'
FOLD_TABLE_PATH = OUTPUT_DIR / 'fold_scores_draft05_xgboost_kaggle.csv'
BLEND_GRID_PATH = OUTPUT_DIR / 'blend_threshold_grid_draft05_xgboost_kaggle.csv'

config_df = pd.DataFrame([
    {
        'competition': COMPETITION,
        'device_mode': DEVICE_MODE,
        'seed': RANDOM_STATE,
        'n_splits': N_SPLITS,
        'threshold_grid_size': len(THRESHOLD_GRID),
        'blend_weight_grid_size': len(BLEND_WEIGHT_GRID),
        'run_download': RUN_DOWNLOAD,
        'run_submission': RUN_SUBMISSION,
        'full_refit_for_submission': FULL_REFIT_FOR_SUBMISSION,
        'output_file': str(OUTPUT_PATH),
    }
])
display(config_df)
print('Train path:', TRAIN_PATH)
print('Test path:', TEST_PATH)
print('Sample path:', SAMPLE_PATH)


In [ ]:
train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)
sample_submission = pd.read_csv(SAMPLE_PATH)

expected_train_cols = {
    'id',
    'time_ind',
    'transac_type',
    'amount',
    'src_acc',
    'src_bal',
    'src_new_bal',
    'dst_acc',
    'dst_bal',
    'dst_new_bal',
    'is_fraud',
    'is_flagged_fraud',
}
expected_test_cols = expected_train_cols - {LABEL_COL}
assert set(train_df.columns) == expected_train_cols, 'Train schema mismatch'
assert set(test_df.columns) == expected_test_cols, 'Test schema mismatch'
assert sample_submission.columns.tolist() == [ID_COL, LABEL_COL], 'Sample submission schema mismatch'

class_counts = train_df[LABEL_COL].value_counts().sort_index()
fraud_rate = float(train_df[LABEL_COL].mean())
print('train shape:', train_df.shape)
print('test shape:', test_df.shape)
print('Class counts (0, 1):')
print(class_counts.to_string())
print(f'Fraud rate: {fraud_rate:.8f} ({fraud_rate * 100:.4f}%)')


In [ ]:
NUMERIC_SOURCE_COLS = ['amount', 'src_bal', 'src_new_bal', 'dst_bal', 'dst_new_bal']

def get_fill_values(fit_df: pd.DataFrame) -> dict[str, float]:
    medians = fit_df[NUMERIC_SOURCE_COLS].median(numeric_only=True).to_dict()
    return {k: float(v) for k, v in medians.items()}

def build_account_maps(fit_df: pd.DataFrame) -> dict[str, pd.Series]:
    src_group = fit_df.groupby('src_acc', observed=True)
    dst_group = fit_df.groupby('dst_acc', observed=True)
    pair_group = fit_df.groupby(['src_acc', 'dst_acc'], observed=True)

    src_df = src_group.agg(
        src_txn_count=(ID_COL, 'size'),
        src_unique_dst=('dst_acc', 'nunique'),
        src_amt_mean=('amount', 'mean'),
        src_amt_median=('amount', 'median'),
        src_amt_max=('amount', 'max'),
        src_amt_std=('amount', 'std'),
        src_type_nunique=('transac_type', 'nunique'),
        src_large_share=('amount', lambda s: float((s > 200_000).mean())),
    )
    dst_df = dst_group.agg(
        dst_txn_count=(ID_COL, 'size'),
        dst_unique_src=('src_acc', 'nunique'),
        dst_amt_mean=('amount', 'mean'),
        dst_amt_median=('amount', 'median'),
        dst_amt_max=('amount', 'max'),
        dst_amt_std=('amount', 'std'),
        dst_type_nunique=('transac_type', 'nunique'),
        dst_large_share=('amount', lambda s: float((s > 200_000).mean())),
    )
    pair_count = pair_group[ID_COL].size().rename('src_dst_pair_count')

    src_df['src_amt_std'] = src_df['src_amt_std'].fillna(0.0)
    dst_df['dst_amt_std'] = dst_df['dst_amt_std'].fillna(0.0)

    return {
        'src_txn_count': src_df['src_txn_count'],
        'src_unique_dst': src_df['src_unique_dst'],
        'src_amt_mean': src_df['src_amt_mean'],
        'src_amt_median': src_df['src_amt_median'],
        'src_amt_max': src_df['src_amt_max'],
        'src_amt_std': src_df['src_amt_std'],
        'src_type_nunique': src_df['src_type_nunique'],
        'src_large_share': src_df['src_large_share'],
        'dst_txn_count': dst_df['dst_txn_count'],
        'dst_unique_src': dst_df['dst_unique_src'],
        'dst_amt_mean': dst_df['dst_amt_mean'],
        'dst_amt_median': dst_df['dst_amt_median'],
        'dst_amt_max': dst_df['dst_amt_max'],
        'dst_amt_std': dst_df['dst_amt_std'],
        'dst_type_nunique': dst_df['dst_type_nunique'],
        'dst_large_share': dst_df['dst_large_share'],
        'src_dst_pair_count': pair_count,
    }

def build_base_features(df: pd.DataFrame, fill_values: dict[str, float]) -> pd.DataFrame:
    out = pd.DataFrame(index=df.index)

    amount = df['amount'].fillna(fill_values['amount']).astype('float64')
    src_bal = df['src_bal'].fillna(fill_values['src_bal']).astype('float64')
    src_new_bal = df['src_new_bal'].fillna(fill_values['src_new_bal']).astype('float64')
    dst_bal = df['dst_bal'].fillna(fill_values['dst_bal']).astype('float64')
    dst_new_bal = df['dst_new_bal'].fillna(fill_values['dst_new_bal']).astype('float64')
    time_ind = df[TIME_COL].astype('int64')

    out['transac_type'] = df['transac_type'].fillna('MISSING').astype(str)
    out['amount'] = amount
    out['src_bal'] = src_bal
    out['src_new_bal'] = src_new_bal
    out['dst_bal'] = dst_bal
    out['dst_new_bal'] = dst_new_bal
    out['is_flagged_fraud'] = df['is_flagged_fraud'].astype('float64')

    out['src_balance_diff'] = src_new_bal - src_bal
    out['dst_balance_diff'] = dst_new_bal - dst_bal
    out['expected_src_new'] = src_bal - amount
    out['src_error'] = out['expected_src_new'] - src_new_bal
    out['expected_dst_new'] = dst_bal + amount
    out['dst_error'] = out['expected_dst_new'] - dst_new_bal
    out['abs_src_error'] = np.abs(out['src_error'])
    out['abs_dst_error'] = np.abs(out['dst_error'])
    out['error_gap'] = out['abs_src_error'] - out['abs_dst_error']
    out['balance_shift_total'] = np.abs(out['src_balance_diff']) + np.abs(out['dst_balance_diff'])

    out['src_zero_before'] = (src_bal == 0).astype('float64')
    out['src_zero_after'] = (src_new_bal == 0).astype('float64')
    out['dst_zero_before'] = (dst_bal == 0).astype('float64')
    out['dst_zero_after'] = (dst_new_bal == 0).astype('float64')

    out['amount_to_src_balance'] = amount / (np.abs(src_bal) + EPS)
    out['amount_to_dst_balance'] = amount / (np.abs(dst_bal) + EPS)
    out['amount_to_src_new_balance'] = amount / (np.abs(src_new_bal) + EPS)
    out['amount_over_src_before'] = amount / (src_bal + EPS)
    out['amount_over_dst_before'] = amount / (dst_bal + EPS)
    out['amount_over_src_before_cap'] = np.clip(out['amount_over_src_before'], -10, 10)
    out['amount_over_dst_before_cap'] = np.clip(out['amount_over_dst_before'], -10, 10)

    out['emptied_account'] = ((src_bal > 0) & (src_new_bal == 0)).astype('float64')
    out['large_transfer'] = (amount > 200_000).astype('float64')
    out['is_round_amount'] = (amount % 1000 == 0).astype('float64')
    out['amount_bucket'] = pd.cut(
        amount,
        bins=[-np.inf, 100, 1_000, 10_000, 100_000, 200_000, 500_000, np.inf],
        labels=False,
    ).astype('float64')

    out['hour'] = (time_ind % 24).astype('float64')
    out['day'] = (time_ind // 24).astype('float64')
    out['is_night'] = out['hour'].isin([0, 1, 2, 3, 4, 5]).astype('float64')
    out['hour_sin'] = np.sin(2 * np.pi * out['hour'] / 24.0)
    out['hour_cos'] = np.cos(2 * np.pi * out['hour'] / 24.0)
    out['log_amount'] = np.log1p(amount)

    src_prefix_merchant = df['src_acc'].astype(str).str.lower().str.startswith('mer')
    dst_prefix_merchant = df['dst_acc'].astype(str).str.lower().str.startswith('mer')
    prefix_merchant = src_prefix_merchant | dst_prefix_merchant
    missing_merchant_proxy = df[['dst_bal', 'dst_new_bal']].isna().any(axis=1)
    out['merchant_prefix_flag'] = prefix_merchant.astype('float64')
    out['merchant_missing_proxy'] = missing_merchant_proxy.astype('float64')
    out['merchant_indicator'] = np.where(prefix_merchant, 1.0, missing_merchant_proxy.astype(float))

    return out

def add_account_features(base_features: pd.DataFrame, df: pd.DataFrame, account_maps: dict[str, pd.Series]) -> pd.DataFrame:
    out = base_features.copy()
    out['src_txn_count'] = df['src_acc'].map(account_maps['src_txn_count']).fillna(0.0).astype('float64')
    out['src_unique_dst'] = df['src_acc'].map(account_maps['src_unique_dst']).fillna(0.0).astype('float64')
    out['src_amt_mean'] = df['src_acc'].map(account_maps['src_amt_mean']).fillna(0.0).astype('float64')
    out['src_amt_median'] = df['src_acc'].map(account_maps['src_amt_median']).fillna(0.0).astype('float64')
    out['src_amt_max'] = df['src_acc'].map(account_maps['src_amt_max']).fillna(0.0).astype('float64')
    out['src_amt_std'] = df['src_acc'].map(account_maps['src_amt_std']).fillna(0.0).astype('float64')
    out['src_type_nunique'] = df['src_acc'].map(account_maps['src_type_nunique']).fillna(0.0).astype('float64')
    out['src_large_share'] = df['src_acc'].map(account_maps['src_large_share']).fillna(0.0).astype('float64')

    out['dst_txn_count'] = df['dst_acc'].map(account_maps['dst_txn_count']).fillna(0.0).astype('float64')
    out['dst_unique_src'] = df['dst_acc'].map(account_maps['dst_unique_src']).fillna(0.0).astype('float64')
    out['dst_amt_mean'] = df['dst_acc'].map(account_maps['dst_amt_mean']).fillna(0.0).astype('float64')
    out['dst_amt_median'] = df['dst_acc'].map(account_maps['dst_amt_median']).fillna(0.0).astype('float64')
    out['dst_amt_max'] = df['dst_acc'].map(account_maps['dst_amt_max']).fillna(0.0).astype('float64')
    out['dst_amt_std'] = df['dst_acc'].map(account_maps['dst_amt_std']).fillna(0.0).astype('float64')
    out['dst_type_nunique'] = df['dst_acc'].map(account_maps['dst_type_nunique']).fillna(0.0).astype('float64')
    out['dst_large_share'] = df['dst_acc'].map(account_maps['dst_large_share']).fillna(0.0).astype('float64')

    pair_keys = list(zip(df['src_acc'], df['dst_acc']))
    pair_count = pd.Series(pair_keys).map(account_maps['src_dst_pair_count'])
    out['src_dst_pair_count'] = pair_count.fillna(0.0).astype('float64')

    out['src_unseen_in_fold'] = (~df['src_acc'].isin(account_maps['src_txn_count'].index)).astype('float64')
    out['dst_unseen_in_fold'] = (~df['dst_acc'].isin(account_maps['dst_txn_count'].index)).astype('float64')
    out['src_dst_unseen_pair'] = (out['src_dst_pair_count'] == 0).astype('float64')
    return out

def make_model_matrix(fit_features: pd.DataFrame, transform_features: pd.DataFrame, encoder: ColumnTransformer | None = None):
    categorical_cols = ['transac_type']
    numeric_cols = [c for c in fit_features.columns if c not in categorical_cols]
    if encoder is None:
        encoder = ColumnTransformer(
            transformers=[
                ('transac_type_ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=True), categorical_cols),
                ('numeric', 'passthrough', numeric_cols),
            ],
            remainder='drop',
            sparse_threshold=1.0,
        )
        fit_matrix = encoder.fit_transform(fit_features)
    else:
        fit_matrix = encoder.transform(fit_features)
    transform_matrix = encoder.transform(transform_features)
    return fit_matrix, transform_matrix, encoder

def has_inf_values(df: pd.DataFrame) -> bool:
    numeric_df = df.select_dtypes(include=[np.number])
    for col in numeric_df.columns:
        if np.isinf(numeric_df[col].to_numpy()).any():
            return True
    return False

def metric_at_threshold(y_true: np.ndarray, prob: np.ndarray, threshold: float) -> dict[str, float]:
    pred = (prob >= threshold).astype(int)
    macro_f1 = f1_score(y_true, pred, average='macro', zero_division=0)
    fraud_f1 = f1_score(y_true, pred, pos_label=1, zero_division=0)
    precision = precision_score(y_true, pred, zero_division=0)
    recall = recall_score(y_true, pred, zero_division=0)
    tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0, 1]).ravel()
    return {
        'macro_f1': float(macro_f1),
        'fraud_f1': float(fraud_f1),
        'precision': float(precision),
        'recall': float(recall),
        'tn': int(tn),
        'fp': int(fp),
        'fn': int(fn),
        'tp': int(tp),
    }

def threshold_search(y_true: np.ndarray, prob: np.ndarray, thresholds: np.ndarray):
    rows = []
    best_threshold = 0.5
    best_macro_f1 = -1.0
    for t in thresholds:
        m = metric_at_threshold(y_true, prob, float(t))
        rows.append({'threshold': float(t), **m})
        if m['macro_f1'] > best_macro_f1:
            best_macro_f1 = m['macro_f1']
            best_threshold = float(t)
    threshold_df = pd.DataFrame(rows)
    best_row = threshold_df.sort_values(['macro_f1', 'threshold'], ascending=[False, True]).iloc[0].to_dict()
    return threshold_df, best_row

def get_feature_importance_df(model: XGBClassifier, encoder: ColumnTransformer) -> pd.DataFrame:
    booster = model.get_booster()
    score_map = booster.get_score(importance_type='gain')
    if not score_map:
        return pd.DataFrame({'feature': [], 'gain': []})
    names = list(encoder.get_feature_names_out())
    key_to_name = {f'f{i}': names[i] for i in range(len(names))}
    rows = []
    for k, v in score_map.items():
        rows.append({'feature': key_to_name.get(k, k), 'gain': float(v)})
    return pd.DataFrame(rows)


In [ ]:
y = train_df[LABEL_COL].astype(int).to_numpy()
oof_a = np.zeros(len(train_df), dtype='float64')
oof_b = np.zeros(len(train_df), dtype='float64')
test_a_folds = []
test_b_folds = []
fold_rows = []
importance_rows = []
best_iters_a = []
best_iters_b = []

skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

for fold, (tr_idx, va_idx) in enumerate(skf.split(train_df, train_df[LABEL_COL]), start=1):
    fold_start = time.time()
    tr_df = train_df.iloc[tr_idx].reset_index(drop=True)
    va_df = train_df.iloc[va_idx].reset_index(drop=True)
    y_tr = tr_df[LABEL_COL].astype(int).to_numpy()
    y_va = va_df[LABEL_COL].astype(int).to_numpy()

    fill_values = get_fill_values(tr_df)
    account_maps = build_account_maps(tr_df)

    X_tr_base = build_base_features(tr_df, fill_values)
    X_va_base = build_base_features(va_df, fill_values)
    X_te_base = build_base_features(test_df, fill_values)

    X_tr = add_account_features(X_tr_base, tr_df, account_maps)
    X_va = add_account_features(X_va_base, va_df, account_maps)
    X_te = add_account_features(X_te_base, test_df, account_maps)

    assert not has_inf_values(X_tr), f'Fold {fold}: inf values in train features'
    assert not has_inf_values(X_va), f'Fold {fold}: inf values in valid features'
    assert not has_inf_values(X_te), f'Fold {fold}: inf values in test features'

    X_tr_mat, X_va_mat, encoder = make_model_matrix(X_tr, X_va, encoder=None)
    _, X_te_mat, _ = make_model_matrix(X_tr, X_te, encoder=encoder)

    va_probe = X_va.head(8).copy()
    va_probe.loc[:, 'transac_type'] = 'UNSEEN_TYPE'
    _ = encoder.transform(va_probe)

    pos_count = int(y_tr.sum())
    neg_count = int(len(y_tr) - pos_count)
    scale_pos_weight = float(neg_count / max(pos_count, 1))

    params_a = {**BASE_MODEL_A, **DEVICE_PARAMS, 'scale_pos_weight': scale_pos_weight}
    params_b = {**BASE_MODEL_B, **DEVICE_PARAMS, 'scale_pos_weight': scale_pos_weight}

    model_a = XGBClassifier(**params_a)
    model_b = XGBClassifier(**params_b)

    model_a.fit(X_tr_mat, y_tr, eval_set=[(X_va_mat, y_va)], verbose=False)
    model_b.fit(X_tr_mat, y_tr, eval_set=[(X_va_mat, y_va)], verbose=False)

    p_va_a = model_a.predict_proba(X_va_mat)[:, 1]
    p_va_b = model_b.predict_proba(X_va_mat)[:, 1]
    p_te_a = model_a.predict_proba(X_te_mat)[:, 1]
    p_te_b = model_b.predict_proba(X_te_mat)[:, 1]

    oof_a[va_idx] = p_va_a
    oof_b[va_idx] = p_va_b
    test_a_folds.append(p_te_a)
    test_b_folds.append(p_te_b)

    it_a = getattr(model_a, 'best_iteration', BASE_MODEL_A['n_estimators'] - 1) + 1
    it_b = getattr(model_b, 'best_iteration', BASE_MODEL_B['n_estimators'] - 1) + 1
    best_iters_a.append(int(it_a))
    best_iters_b.append(int(it_b))

    m_a = metric_at_threshold(y_va, p_va_a, 0.5)
    m_b = metric_at_threshold(y_va, p_va_b, 0.5)
    m_blend05 = metric_at_threshold(y_va, 0.5 * p_va_a + 0.5 * p_va_b, 0.5)

    fold_seconds = float(time.time() - fold_start)
    fold_rows.append({
        'fold': fold,
        'valid_rows': int(len(va_df)),
        'valid_fraud_rate': float(va_df[LABEL_COL].mean()),
        'macro_f1_a_at_0_5': m_a['macro_f1'],
        'macro_f1_b_at_0_5': m_b['macro_f1'],
        'macro_f1_blend_0_5w_at_0_5': m_blend05['macro_f1'],
        'best_iter_a': int(it_a),
        'best_iter_b': int(it_b),
        'scale_pos_weight': scale_pos_weight,
        'fold_seconds': fold_seconds,
    })

    imp_a = get_feature_importance_df(model_a, encoder)
    if not imp_a.empty:
        imp_a['fold'] = fold
        imp_a['model'] = 'A'
        importance_rows.append(imp_a)
    imp_b = get_feature_importance_df(model_b, encoder)
    if not imp_b.empty:
        imp_b['fold'] = fold
        imp_b['model'] = 'B'
        importance_rows.append(imp_b)

    print(f'Fold {fold}/{N_SPLITS} done | A={m_a["macro_f1"]:.6f} B={m_b["macro_f1"]:.6f} blend0.5={m_blend05["macro_f1"]:.6f} | {fold_seconds:.1f}s')

assert np.isfinite(oof_a).all() and not np.isnan(oof_a).any(), 'OOF A invalid values'
assert np.isfinite(oof_b).all() and not np.isnan(oof_b).any(), 'OOF B invalid values'

fold_scores_df = pd.DataFrame(fold_rows)
display(fold_scores_df)
print('Median best_iter A:', int(np.median(best_iters_a)))
print('Median best_iter B:', int(np.median(best_iters_b)))


In [ ]:
blend_rows = []
for w in BLEND_WEIGHT_GRID:
    p = w * oof_a + (1.0 - w) * oof_b
    m05 = metric_at_threshold(y, p, 0.5)
    blend_rows.append({
        'blend_weight_a': float(w),
        'blend_weight_b': float(1.0 - w),
        'macro_f1_at_0_5': m05['macro_f1'],
        'fraud_f1_at_0_5': m05['fraud_f1'],
        'precision_at_0_5': m05['precision'],
        'recall_at_0_5': m05['recall'],
    })

blend_df = pd.DataFrame(blend_rows).sort_values('macro_f1_at_0_5', ascending=False).reset_index(drop=True)
best_w = float(blend_df.iloc[0]['blend_weight_a'])
oof_blend = best_w * oof_a + (1.0 - best_w) * oof_b

thresh_df, best_thresh_row = threshold_search(y, oof_blend, THRESHOLD_GRID)
best_threshold = float(best_thresh_row['threshold'])

baseline_prob = oof_a.copy()
baseline_05 = metric_at_threshold(y, baseline_prob, 0.5)
_, baseline_best_row = threshold_search(y, baseline_prob, THRESHOLD_GRID)
baseline_best_macro = float(baseline_best_row['macro_f1'])

opt_05 = metric_at_threshold(y, oof_blend, 0.5)
opt_best = metric_at_threshold(y, oof_blend, best_threshold)

assert opt_best['macro_f1'] + 1e-12 >= baseline_best_macro, 'Optimized blend should be >= baseline single-model best threshold macro F1'
assert opt_best['macro_f1'] + 1e-12 >= baseline_05['macro_f1'], 'Optimized blend should be >= baseline single-model @0.5 macro F1'
assert opt_best['macro_f1'] + 1e-12 >= opt_05['macro_f1'], 'Optimized threshold should be >= blend @0.5 macro F1'

print('Best blend weight A:', best_w)
print('Baseline A @0.5:', baseline_05)
print('Baseline A best-threshold macro F1:', baseline_best_macro)
print('Blend @0.5:', opt_05)
print('Blend best threshold:', {'threshold': best_threshold, **opt_best})

blend_threshold_df = blend_df.merge(
    pd.DataFrame([{'best_threshold': best_threshold, 'best_macro_f1_after_threshold': opt_best['macro_f1'], 'selected_blend_weight_a': best_w}]),
    how='cross',
)
display(blend_df.head(10))
display(thresh_df.sort_values('macro_f1', ascending=False).head(10))

if importance_rows:
    imp_all = pd.concat(importance_rows, ignore_index=True)
    imp_mean = imp_all.groupby(['model', 'feature'], as_index=False)['gain'].mean().sort_values(['model', 'gain'], ascending=[True, False])
else:
    imp_mean = pd.DataFrame({'model': [], 'feature': [], 'gain': []})
print('Top feature importances by model:')
display(imp_mean.groupby('model').head(20))


In [ ]:
if FULL_REFIT_FOR_SUBMISSION:
    full_fill = get_fill_values(train_df)
    full_maps = build_account_maps(train_df)
    X_full_base = build_base_features(train_df, full_fill)
    X_test_base = build_base_features(test_df, full_fill)
    X_full = add_account_features(X_full_base, train_df, full_maps)
    X_test = add_account_features(X_test_base, test_df, full_maps)
    assert not has_inf_values(X_full), 'Inf values in full train features'
    assert not has_inf_values(X_test), 'Inf values in full test features'
    X_full_mat, X_test_mat, enc_full = make_model_matrix(X_full, X_test, encoder=None)
    y_full = train_df[LABEL_COL].astype(int).to_numpy()
    pos = int(y_full.sum())
    neg = int(len(y_full) - pos)
    spw = float(neg / max(pos, 1))
    cap_a = int(np.median(best_iters_a))
    cap_b = int(np.median(best_iters_b))
    refit_a = {**BASE_MODEL_A, **DEVICE_PARAMS, 'scale_pos_weight': spw, 'n_estimators': cap_a}
    refit_b = {**BASE_MODEL_B, **DEVICE_PARAMS, 'scale_pos_weight': spw, 'n_estimators': cap_b}
    refit_a.pop('early_stopping_rounds', None)
    refit_b.pop('early_stopping_rounds', None)
    m_a = XGBClassifier(**refit_a)
    m_b = XGBClassifier(**refit_b)
    m_a.fit(X_full_mat, y_full, verbose=False)
    m_b.fit(X_full_mat, y_full, verbose=False)
    test_prob_a = m_a.predict_proba(X_test_mat)[:, 1]
    test_prob_b = m_b.predict_proba(X_test_mat)[:, 1]
else:
    test_prob_a = np.mean(np.column_stack(test_a_folds), axis=1)
    test_prob_b = np.mean(np.column_stack(test_b_folds), axis=1)

test_prob_blend = best_w * test_prob_a + (1.0 - best_w) * test_prob_b
test_pred = (test_prob_blend >= best_threshold).astype(int)
submission = pd.DataFrame({ID_COL: test_df[ID_COL], LABEL_COL: test_pred})

assert len(submission) == len(test_df), 'Submission length mismatch'
assert submission.columns.tolist() == [ID_COL, LABEL_COL], 'Submission schema mismatch'
assert set(submission[LABEL_COL].unique()).issubset({0, 1}), 'Submission target must be binary'

submission.to_csv(OUTPUT_PATH, index=False)
fold_scores_df.to_csv(FOLD_TABLE_PATH, index=False)
blend_threshold_df.to_csv(BLEND_GRID_PATH, index=False)

diagnostics = pd.DataFrame([
    {
        'train_rows': int(len(train_df)),
        'test_rows': int(len(test_df)),
        'device_mode': DEVICE_MODE,
        'full_refit_for_submission': FULL_REFIT_FOR_SUBMISSION,
        'baseline_a_macro_f1_at_0_5': float(baseline_05['macro_f1']),
        'baseline_a_macro_f1_best_threshold': float(baseline_best_macro),
        'blend_macro_f1_at_0_5': float(opt_05['macro_f1']),
        'blend_macro_f1_best': float(opt_best['macro_f1']),
        'blend_fraud_f1_best': float(opt_best['fraud_f1']),
        'blend_precision_best': float(opt_best['precision']),
        'blend_recall_best': float(opt_best['recall']),
        'blend_weight_a': float(best_w),
        'blend_weight_b': float(1.0 - best_w),
        'best_threshold': float(best_threshold),
        'predicted_fraud_rows_test': int(submission[LABEL_COL].sum()),
        'predicted_fraud_rate_test': float(submission[LABEL_COL].mean()),
        'median_best_iter_a': int(np.median(best_iters_a)),
        'median_best_iter_b': int(np.median(best_iters_b)),
        'mean_fold_seconds': float(fold_scores_df['fold_seconds'].mean()),
    }
])
diagnostics.to_csv(DIAGNOSTICS_PATH, index=False)
display(diagnostics)
display(submission.head())
print('Saved submission:', OUTPUT_PATH)
print('Saved diagnostics:', DIAGNOSTICS_PATH)
print('Saved fold scores:', FOLD_TABLE_PATH)
print('Saved blend grid:', BLEND_GRID_PATH)


In [ ]:
if RUN_SUBMISSION:
    if api is None:
        raise RuntimeError('RUN_SUBMISSION=True but Kaggle API auth is not available')
    submit_message = (
        f'draft-05 xgb blend | macro_f1={opt_best["macro_f1"]:.6f} '
        f'fraud_f1={opt_best["fraud_f1"]:.6f} '
        f'wA={best_w:.2f} thr={best_threshold:.3f} '
        f'time={datetime.now(UTC).strftime("%Y-%m-%d %H:%M:%S")}Z'
    )
    response = api.competition_submit(
        file_name=str(OUTPUT_PATH),
        message=submit_message,
        competition=COMPETITION,
        quiet=False,
    )
    print('Submission response:', response)
else:
    print('RUN_SUBMISSION is False. File is ready at', OUTPUT_PATH)


## Draft-05 notes

We kept XGBoost only, but used two different model profiles and blended their OOF probabilities.
Then we tuned threshold on blended OOF for Macro F1, which matches the competition metric.
This setup usually gives more stable ranking than a single model, especially with extreme class imbalance.
